# Full Bayesian Operator Inference: FitzHugh-Nagumo

**Workflow:**
1. Generate training data and fit POD basis
2. Grid search for prior operator
3. Fit GP hyperparameters
4. Run Bayesian inference (SVI/MCMC)
5. Evaluate and visualize results

## 1. Setup and Configuration

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import jax
import jax.numpy as jnp
import opinf
import matplotlib.pyplot as plt
import numpyro
import numpyro.distributions as dist

from core import (
    generate_trajectory,
    JaxCompatibleModel,
    grid_search_prior_operator,
    fit_gp_hyperparameters_mle,
    compute_gp_derivatives,
    build_bayesian_opinf_model,
    run_svi,
    run_mcmc,
    generate_rom_predictions,
    plot_deterministic_rom_solves,
    plot_gp_fit,
    plot_full_order_error,
    Plotter,
    run_diagnostics,
    DataScaler,
)
import config
from config import Basis, ReducedOrderModel
from fitz_plotter import FitzPlotter

numpyro.set_platform('cpu')
numpyro.set_host_device_count(4)

# Reproducibility
np.random.seed(42)
rng_key = jax.random.PRNGKey(42)

In [ ]:
# === EXPERIMENT CONFIGURATION ===
OPERATORS = "cAHBN"       # OpInf operator structure (with inputs)
NUM_MODES = 4             # POD modes to retain

# Data generation settings
TRAINING_SPAN = (0, 3.0)  # Subset of time domain for training
PREDICTION_SPAN = (0, 5.0) # Full time domain for ROM evaluation
NUM_SAMPLES = 300          # Number of training samples
NOISE_LEVEL = 0.01        # Noise level for training data
NUM_EVAL_POINTS = None    # Set to None for no densification

# Data scaling
USE_SCALED_DATA = True    # Standardize POD coefficients for GP fitting

# Inference settings
RUN_SVI = True
RUN_MCMC = False           # More expensive, optional
USE_SVI_FOR_MCMC_INIT = True  # Initialize MCMC from SVI result
GUIDE = numpyro.infer.autoguide.AutoDelta  # Guide for SVI

# Hyperparameters
GAMMA = 1e-1     # Operator prior variance
GAMMA2 = 1e-1    # ODE constraint stiffness
GAMMA3 = None    # Operator prior variance for MCMC (if different from SVI)
GAMMA4 = None    # ODE constraint stiffness for MCMC (if different from SVI)

## 2. Generate Training Data

In [ ]:
# Generate training data using the standard pipeline
(fom, time_domain_full, true_states, time_sampled, snapshots_sampled) = \
    generate_trajectory(config, TRAINING_SPAN, NUM_SAMPLES, NOISE_LEVEL)

# Input function
input_func = ReducedOrderModel.input_func
inputs_sampled = np.array(input_func(time_sampled))

print(f"Full time domain: {time_domain_full.shape}")
print(f"True states: {true_states.shape}")
print(f"Sampled times: {time_sampled.shape}")
print(f"Sampled snapshots: {snapshots_sampled.shape}")
print(f"Training span: [{TRAINING_SPAN[0]:.4f}, {TRAINING_SPAN[1]:.4f}]")
print(f"Prediction span: [{PREDICTION_SPAN[0]:.4f}, {PREDICTION_SPAN[1]:.4f}]")

In [ ]:
# Fit POD basis
basis = Basis(num_vectors=NUM_MODES)
basis.fit(snapshots_sampled)

snapshots_comp_sampled = basis.compress(snapshots_sampled)
full_states_compressed = basis.compress(true_states)
print(f"Compressed shape: {snapshots_comp_sampled.shape}")
print(f"Cumulative energy: {basis.cumulative_energy:.4%}")

In [ ]:
# Set up evaluation time domains for plotting
NUM_REGRESSION_POINTS = 150
time_domain_eval_training = np.linspace(TRAINING_SPAN[0], TRAINING_SPAN[1], NUM_REGRESSION_POINTS)
time_domain_eval_prediction = np.linspace(PREDICTION_SPAN[0], PREDICTION_SPAN[1], NUM_REGRESSION_POINTS)

print(f"Training samples: {time_sampled.shape[0]}")
print(f"Eval training domain: [{time_domain_eval_training[0]:.4f}, {time_domain_eval_training[-1]:.4f}] ({len(time_domain_eval_training)} pts)")
print(f"Eval prediction domain: [{time_domain_eval_prediction[0]:.4f}, {time_domain_eval_prediction[-1]:.4f}] ({len(time_domain_eval_prediction)} pts)")

## 3a. Data Scaling (Optional)

In [ ]:
# Optionally scale data for GP fitting
if USE_SCALED_DATA:
    data_scaler = DataScaler(num_modes=NUM_MODES)
    data_scaler.fit(snapshots_comp_sampled)
    training_data = data_scaler.transform(snapshots_comp_sampled)
    print(f"Scaling enabled: {data_scaler}")
else:
    data_scaler = None
    training_data = snapshots_comp_sampled
    print("Scaling disabled: using raw POD coefficients")

## 3. Grid Search for Prior Operator

In [ ]:
# Find best deterministic operator via regularization grid search
result = grid_search_prior_operator(
    basis=basis,
    time_domain_sampled=time_sampled,
    snapshots_sampled=snapshots_sampled,
    snapshots_compressed=snapshots_comp_sampled,
    operators=OPERATORS,
    inputs=inputs_sampled.T if inputs_sampled.ndim > 1 else inputs_sampled[:, None].T,
    input_func=input_func,
    verbose=True
)

prior_operator = result.operator
rom = result.rom
print(f"\nPrior operator shape: {prior_operator.shape}")

In [ ]:
# Debug: Plot all stable deterministic ROM solves from grid search
fig, axes = plot_deterministic_rom_solves(
    grid_search_result=result,
    snapshots_compressed=snapshots_comp_sampled,
    time_sampled=time_sampled,
    time_eval_training=time_domain_eval_training,
    time_eval_prediction=time_domain_eval_prediction,
    time_full=time_domain_full,
    true_states_compressed=full_states_compressed,
)
plt.show()

## 4. Fit GP Hyperparameters

In [ ]:
# Fit GP for each mode via MLE (on scaled data if enabled)
Ls, Vs, Ns, gp_models = fit_gp_hyperparameters_mle(
    time_domain=time_sampled,
    snapshots=training_data,
    verbose=True
)

In [ ]:
# Visualize GP fit quality (states and derivatives)
fig, axes = plot_gp_fit(
    gp_models=gp_models,
    snapshots_compressed=training_data,
    time_sampled=time_sampled,
    time_eval=time_domain_eval_training,
    lengthscales=Ls,
    variances=Vs,
    plot_derivatives=True,
)
plt.suptitle(f"GP Fit ({'scaled' if USE_SCALED_DATA else 'unscaled'} data)", y=1.02)
plt.show()

## 5. Bayesian Inference

In [ ]:
# Build evaluation time grid (densified via GP if NUM_EVAL_POINTS is set)
if NUM_EVAL_POINTS is not None:
    time_eval = np.linspace(time_sampled[0], time_sampled[-1], NUM_EVAL_POINTS)
    print(f"GP densification: evaluating ODE constraints at {NUM_EVAL_POINTS} points "
          f"(vs {len(time_sampled)} training samples)")
else:
    time_eval = time_sampled
    print(f"No densification: evaluating at {len(time_sampled)} training times")

# Precompute GP mean predictions at evaluation points
Xs_means = np.array([
    gp_models[i].predict(time_eval[:, None], return_std=False)
    for i in range(NUM_MODES)
])
inputs_eval = np.array(input_func(time_eval)).T

# Build Bayesian model using shared factory function
bayesian_opinf_model = build_bayesian_opinf_model(
    prior_operator=prior_operator,
    rom=rom,
    Ls_means=Ls,
    Vs_means=Vs,
    time_domain_sampled=time_sampled,
    snapshots=training_data,
    Xs_means=Xs_means,
    Ns_means=Ns,
    inputs_eval=inputs_eval,
    data_scaler=data_scaler if USE_SCALED_DATA else None,
)
print("Bayesian model built")

In [ ]:
# Run SVI
if RUN_SVI:
    svi_result = run_svi(
        model=bayesian_opinf_model,
        rng_key=rng_key,
        time_eval=time_eval,
        gamma=GAMMA,
        gamma2=GAMMA2,
        num_steps=30000,
        learning_rate=1e-4,
        verbose=True,
        guide=GUIDE
    )
    samples = svi_result.samples
    
    # Extract operator for potential MCMC initialization
    O_key = next((k for k in samples.keys() if 'O' in k and 'ode' not in k.lower()), None)
    if O_key:
        svi_O = samples[O_key]
        svi_O_init = svi_O.mean(axis=0) if svi_O.ndim > 2 else svi_O[0]
        print(f"SVI operator extracted from key '{O_key}', shape: {svi_O_init.shape}")
    else:
        svi_O_init = None
        print("Warning: Could not extract O from SVI samples")
    
    # Plot loss
    plt.figure(figsize=(10, 4))
    plt.plot(svi_result.losses, lw=4)
    plt.xlabel('Iteration')
    plt.ylabel('ELBO Loss')
    plt.title('SVI Convergence')
    plt.show()

### 5a. SVI Results

In [ ]:
import importlib, fitz_plotter
importlib.reload(fitz_plotter)
from fitz_plotter import FitzPlotter

def _evaluate_and_plot(stage_samples, stage_name):
    """Generate predictions and plots for a given inference stage."""
    Os_stage, Xs_stage, rom_solves_stage = generate_rom_predictions(
        samples=stage_samples, rom=rom,
        snapshots_compressed=snapshots_comp_sampled,
        time_eval=time_domain_eval_prediction,
        num_modes=NUM_MODES, num_pulls=200,
        input_func=input_func,
        data_scaler=data_scaler if USE_SCALED_DATA else None,
    )
    print(f"[{stage_name}] Operator samples: {len(Os_stage)}, "
          f"Stable solves: {len(rom_solves_stage)}")

    # Operator posterior summary
    O_mean_stage = Os_stage.mean(axis=0)
    O_std_stage = Os_stage.std(axis=0)
    print(f"\n{stage_name} Posterior mean (first row): {O_mean_stage[0, :5]}")
    print(f"{stage_name} Posterior std  (first row): {O_std_stage[0, :5]}")

    # Get latent state samples and inverse-transform if scaled
    latent_state_samples = []
    for i in range(NUM_MODES):
        key_multi = f'X0_{i}'
        key_legacy = f'X{i}'
        key = key_multi if key_multi in stage_samples else key_legacy
        if key in stage_samples:
            Xi = stage_samples[key]
            if USE_SCALED_DATA and data_scaler is not None:
                Xi = np.array(Xi) * data_scaler.stds_[i, 0] + data_scaler.means_[i, 0]
            latent_state_samples.append(Xi)
        else:
            latent_state_samples.append(np.tile(snapshots_comp_sampled[i], (len(Os_stage), 1)))

    # State reconstruction using FitzPlotter
    plotter_stage = FitzPlotter(
        numPODmodes=NUM_MODES,
        time_domain_training=time_sampled,
        time_domain_prediction=time_domain_full,
        time_domain_eval_training=time_domain_eval_training,
        time_domain_eval_prediction=time_domain_eval_prediction,
        snapshots_training=snapshots_comp_sampled,
        snapshots_prediction=full_states_compressed,
    )

    # Single-column view
    plotter_stage.operator_plot(
        q0=snapshots_comp_sampled[:, 0],
        operator_samples=Os_stage,
        latent_state_samples=latent_state_samples,
        rom=rom,
        input_func=input_func,
        figsize=(10, 2.5 * NUM_MODES),
        max_num_samples=200,
        plot_single=True,
    )
    plt.suptitle(f'ROM Predictions ({stage_name})', fontsize=14)
    plt.show()

    # Compute errors
    if len(rom_solves_stage) > 0:
        errors = []
        for sol in rom_solves_stage:
            err = np.linalg.norm(sol - snapshots_comp_sampled) / np.linalg.norm(snapshots_comp_sampled)
            errors.append(err)
        print(f"  Mean relative error: {np.mean(errors):.4%}")
        print(f"  Stable solves: {len(rom_solves_stage)}")

    return Os_stage, rom_solves_stage

In [ ]:
# === SVI Results ===
if RUN_SVI:
    svi_Os, svi_rom_solves = _evaluate_and_plot(svi_result.samples, "SVI")

In [ ]:
# Run MCMC (optional, more expensive)
if RUN_MCMC:
    # Initialize from SVI if requested and available
    init_values = None
    if USE_SVI_FOR_MCMC_INIT and RUN_SVI and 'svi_O_init' in dir() and svi_O_init is not None:
        init_values = {"O": svi_O_init}
        print(f"Initializing MCMC from SVI result (shape: {svi_O_init.shape})")
    else:
        print("MCMC will find its own starting point via NUTS adaptation")
    
    mcmc_result = run_mcmc(
        model=bayesian_opinf_model,
        rng_key=jax.random.PRNGKey(1),
        time_eval=time_eval,
        init_values=init_values,
        gamma=GAMMA3 if GAMMA3 else GAMMA,
        gamma2=GAMMA4 if GAMMA4 else GAMMA2,
        num_warmup=500,
        num_samples=500,
        num_chains=2,
        verbose=True
    )
    samples = mcmc_result.samples

### 5b. MCMC Results

In [ ]:
# === MCMC Results ===
if RUN_MCMC:
    mcmc_Os, mcmc_rom_solves = _evaluate_and_plot(mcmc_result.samples, "MCMC")

## 6. Summary

In [ ]:
# Summary
print("=" * 50)
print("EXPERIMENT SUMMARY: FitzHugh-Nagumo")
print("=" * 50)
print(f"Operators: {OPERATORS}")
print(f"Modes: {NUM_MODES}")
print(f"Training span: [{TRAINING_SPAN[0]}, {TRAINING_SPAN[1]}]")
print(f"Prediction span: [{PREDICTION_SPAN[0]}, {PREDICTION_SPAN[1]}]")
print(f"Scaling: {'enabled' if USE_SCALED_DATA else 'disabled'}")
print(f"Prior regularization: {result.best_reg:.1e}")
print(f"Prior error: {result.best_error:.4%}")
print(f"Gamma (operator): {GAMMA}")
print(f"Gamma2 (ODE): {GAMMA2}")
print(f"Guide: {GUIDE.__name__}")
if RUN_SVI:
    print(f"\nSVI: {len(svi_Os)} operator samples, {len(svi_rom_solves)} stable solves")
if RUN_MCMC:
    print(f"MCMC: {len(mcmc_Os)} operator samples, {len(mcmc_rom_solves)} stable solves")

## 7. Bayesian Model Diagnostics

Run post-inference diagnostics to check for common pitfalls:
- **Posterior correlation** — highly correlated parameters suggest redundancy or non-identifiability
- **Effective sample size (ESS)** — low ESS means samples are highly autocorrelated
- **Prior-posterior overlap** — checks if data is actually informing the posterior
- **Trace plots** — visual check for mixing and stationarity
- **Divergences & R-hat** — MCMC-specific convergence checks (when applicable)

In [ ]:
# Use MCMC samples if available, otherwise SVI
diag_samples = mcmc_result.samples if RUN_MCMC else svi_result.samples

# Run full diagnostic suite
diag_report = run_diagnostics(
    samples=diag_samples,
    param_name="O",
    prior_mean=prior_operator,
    prior_std=GAMMA,
    mcmc_result=mcmc_result if RUN_MCMC else None,
    samples_by_chain=None,
    correlation_threshold=0.9,
    ess_threshold=100,
    verbose=True,
    plot=True,
)